# Render RSTB (Tahap 7.8) di Google Colab

Data simulasi studi kasus portofolio. Jalankan sel berurutan. **Runtime → Change runtime type → T4 GPU** dulu.

1. Unggah `RSTB-colab.zip` ke **My Drive** (akar Drive).
2. Hasil (PNG, frame, MP4) tersimpan langsung di `My Drive/RSTB-hasil/`, jadi aman bila sesi putus: jalankan ulang sel 1–4, lalu lanjutkan sel yang belum selesai (frame yang sudah ada dilewati).

In [ ]:
# 1. Cek GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. Sambungkan Google Drive, salin dan ekstrak paket ke disk Colab
from google.colab import drive
drive.mount('/content/drive')
!cp -n "/content/drive/MyDrive/RSTB-colab.zip" /content/ && cd /content && unzip -q -o RSTB-colab.zip && echo PAKET SIAP
!mkdir -p "/content/drive/MyDrive/RSTB-hasil" && ln -sfn "/content/drive/MyDrive/RSTB-hasil" "/content/RSTB-colab/07 DED dan RKS" && ls /content/RSTB-colab

In [ ]:
# 3. Pasang Blender 5.2.4 LTS (Linux) dan pustaka sistem
!cd /content && [ -d blender-5.2.4-linux-x64 ] || (wget -q https://download.blender.org/release/Blender5.2/blender-5.2.4-linux-x64.tar.xz && tar -xf blender-5.2.4-linux-x64.tar.xz)
!apt-get -qq install -y libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libegl1 libsm6 libxkbcommon0 fonts-liberation > /dev/null 2>&1
!/content/blender-5.2.4-linux-x64/blender -b --version | head -1

In [ ]:
# 4. Fungsi bantu
import subprocess, glob, os
B = '/content/blender-5.2.4-linux-x64/blender'
AKAR = '/content/RSTB-colab'
def render(kamera, *ekstra):
    cmd = [B, '-b', f'{AKAR}/render/RSTB-Dasar-T7.blend', '--python', f'{AKAR}/skrip/tahap7_render.py', '--', '--kamera', kamera, *ekstra]
    p = subprocess.run(cmd, cwd=AKAR, capture_output=True, text=True)
    baris = [b for b in p.stdout.splitlines() if any(k in b for k in ('SELESAI', 'Traceback', 'Error', 'DETAIL', 'WAKTU'))]
    print(kamera, '
'.join(baris[-6:]) or p.stdout[-1500:])

In [ ]:
# 5. Gambar diam R-01..R-08 (3.200 x 1.800 px), +-3-5 menit per gambar di T4
for k in ['R-01', 'R-02', 'R-03', 'R-04', 'R-05', 'R-06', 'R-07', 'R-08']:
    if os.path.exists(f'{AKAR}/07 DED dan RKS/Render/{k}.png'):
        print(k, 'sudah ada'); continue
    render(k)

In [ ]:
# 6. Uji video: satu shot kecil dengan Eevee (640 px, tiap frame ke-6). Bila gagal/hitam, pakai CYCLES = True di sel 7
render('V-01', '--uji')
print(sorted(glob.glob(f'{AKAR}/07 DED dan RKS/Video/frame/V-01 uji/*.png'))[:3])

In [ ]:
# 7. Video penuh 6 shot (1920 x 1080, 24 fps). Eevee +-1-2 jam; Cycles +-3-5 jam (bisa dilanjutkan bila sesi putus)
CYCLES = False
for k in ['V-01', 'V-02', 'V-03', 'V-04', 'V-05', 'V-06']:
    render(k, *(['--video-cycles'] if CYCLES else []))

In [ ]:
# 8. Rakit MP4 (judul, transisi silang, bingkai 2,35:1) -> My Drive/RSTB-hasil/Video/RSTB-Sinematik-T7.mp4
!cd /content/RSTB-colab && python3 skrip/tahap7_video.py